In [ ]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import HOG, ColorMoments
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### Color moment features

In [28]:
cm = ColorMoments(grid=(8, 8)).fit(X_tr)

X_tr_cm = cm.transform(X_tr)
X_te_cm = cm.transform(X_te)

X_train_cm, X_val_cm, y_train_cm, y_val_cm = train_val_split(
    X_tr_cm, y_tr, test_size=0.3, seed=20
)

In [29]:
X_train_cm.shape

(3500, 576)

In [30]:
lams = [1e-3, 5e-4, 1e-4, 5e-5]

gamma = estimate_laplacian_gamma(X_train_cm)

best = {"acc": -1.0, "lam": None}

for lam in lams:
    kernel_fn = partial(
        laplacian_kernel_matrix,
        gamma=gamma,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_cm, y_train_cm, X_val=X_val_cm, y_val=y_val_cm)

    pred_val, _ = model.predict(X_val_cm)
    acc = accuracy(y_val_cm, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-03, val_acc=0.3860
lam=5.0e-04, val_acc=0.3920
lam=1.0e-04, val_acc=0.4020
lam=5.0e-05, val_acc=0.3953
best: {'acc': 0.402, 'lam': 0.0001}


### HOG features

In [31]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

Grid search over kernel weights

In [33]:
gamma_cm = estimate_laplacian_gamma(X_train_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)

gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# Two-kernel setup:
specs: list[KernelSpec] = [
    {
        "name": "cm_laplacian",
        "Z": X_train_cm,
        "kernel_fn": cm_laplacian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

out = beta_grid_search_cv_two_kernels(
    specs,
    y_train_hog,
    n_classes=n_classes,
    model="krr",
    betas=np.linspace(0, 1, 21),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print("best_beta:", out["best_beta"], "best_mean_acc:", out["best_mean_acc"])

beta=0.000 mean_acc=0.5489
beta=0.050 mean_acc=0.5637
beta=0.100 mean_acc=0.5723
beta=0.150 mean_acc=0.5769
beta=0.200 mean_acc=0.5834
beta=0.250 mean_acc=0.5849
beta=0.300 mean_acc=0.5866
beta=0.350 mean_acc=0.5886
beta=0.400 mean_acc=0.5877
beta=0.450 mean_acc=0.5880
beta=0.500 mean_acc=0.5831
beta=0.550 mean_acc=0.5786
beta=0.600 mean_acc=0.5800
beta=0.650 mean_acc=0.5783
beta=0.700 mean_acc=0.5729
beta=0.750 mean_acc=0.5674
beta=0.800 mean_acc=0.5614
beta=0.850 mean_acc=0.5529
beta=0.900 mean_acc=0.5377
beta=0.950 mean_acc=0.5051
beta=1.000 mean_acc=0.3800
best_beta: 0.3499999940395355 best_mean_acc: 0.5885714285714286


## After tuning parameters, train the best model.

### Check the best model

In [32]:
beta = float(np.asarray(out["best_beta"]).item())

# IMPORTANT: use the partial kernels you defined (with gammas)
Kc_tr = cm_laplacian(X_train_cm, X_train_cm)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)

# Match CV: unit-diagonal normalisation per kernel
Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_train_cm))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))

Ktr = beta * Kc_tr + (1.0 - beta) * Kh_tr

Ks_val = cm_laplacian(X_val_cm, X_train_cm)
Kh_val = hog_chi2(X_val_hog, X_train_hog)

Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_cm), unit_diag(X_train_cm))
Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))

K_val = beta * Ks_val + (1.0 - beta) * Kh_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_cm, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.5606666666666666


### Train on full dataset for submission

In [35]:
beta = float(np.asarray(out["best_beta"]).item())

# Re-estimate gammas on FULL training set (not the CV subset)
gamma_cm = estimate_laplacian_gamma(X_tr_cm)
cm_laplacian = partial(laplacian_kernel_matrix, gamma=gamma_cm)

gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# FULL train Gram
Kc_tr = cm_laplacian(X_tr_cm, X_tr_cm)
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)

Kc_tr = normalise_train_gram(Kc_tr, unit_diag(X_tr_cm))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))

Ktr = beta * Kc_tr + (1.0 - beta) * Kh_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Gram
Ks_te_tr = cm_laplacian(X_te_cm, X_tr_cm)
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)

Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_cm), unit_diag(X_tr_cm))
Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))

K_star = beta * Ks_te_tr + (1.0 - beta) * Kh_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")